In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

img = cv2.imread("scenery.jpg", 0)

noisy = img.copy()
prob = 0.2
rnd = np.random.rand(*img.shape)
noisy[rnd < prob / 2] = 0
noisy[(rnd >= prob / 2) & (rnd < prob)] = 255

s_init = int(input("Enter initial window size (e.g. 3): "))
s_max = int(input("Enter maximum window size Smax (e.g. 7): "))

h, w = img.shape
pad_max = s_max // 2
padded = cv2.copyMakeBorder(noisy, pad_max, pad_max, pad_max, pad_max, cv2.BORDER_REPLICATE)
adapt_output = np.zeros_like(noisy)

for i in range(h):
    for j in range(w):
        pi, pj = i + pad_max, j + pad_max
        z_xy = int(padded[pi, pj])
        s = s_init
        while s <= s_max:
            half = s // 2
            win = padded[pi-half : pi+half+1, pj-half : pj+half+1]
            z_min = int(np.min(win))
            z_max = int(np.max(win))
            z_med = int(np.median(win))
            
            if z_min < z_med < z_max:
                if z_min < z_xy < z_max:
                    adapt_output[i, j] = z_xy
                else:
                    adapt_output[i, j] = z_med
                break
            else:
                s += 2
                if s > s_max:
                    adapt_output[i, j] = z_med
                    break

std_median = cv2.medianBlur(noisy, s_init)

def get_psnr(orig, filtered):
    mse = np.mean((orig.astype(float) - filtered.astype(float)) ** 2)
    return 10 * np.log10((255**2) / mse) if mse > 0 else 100

psnr_std = get_psnr(img, std_median)
psnr_adapt = get_psnr(img, adapt_output)

print(f"PSNR of Standard Median: {psnr_std:.2f} dB")
print(f"PSNR of Adaptive Median: {psnr_adapt:.2f} dB")

plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
plt.imshow(noisy, cmap='gray')
plt.title("Noisy Image")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(std_median, cmap='gray')
plt.title(f"Standard Median ({psnr_std:.2f} dB)")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(adapt_output, cmap='gray')
plt.title(f"Adaptive Median ({psnr_adapt:.2f} dB)")
plt.axis("off")

plt.show()


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

img = cv2.imread("scenery.jpg", 0)

hist, _ = np.histogram(img.flatten(), 256, [0, 256])
p = hist / img.size
i_vals = np.arange(256)
mu_T = np.sum(i_vals * p)

max_sigma = -1
best_t = 0
omega0 = 0
mu_k = 0

for t in range(256):
    omega0 += p[t]
    mu_k += t * p[t]
    omega1 = 1 - omega0
    if omega0 == 0 or omega1 == 0:
        continue
    mu0 = mu_k / omega0
    mu1 = (mu_T - mu_k) / omega1
    sigma_b = omega0 * omega1 * ((mu0 - mu1) ** 2)
    if sigma_b > max_sigma:
        max_sigma = sigma_b
        best_t = t

segmented = np.where(img >= best_t, 255, 0).astype(np.uint8)

print("Optimal Otsu Threshold:", best_t)

plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
plt.imshow(img, cmap='gray')
plt.title("Original Grayscale")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.hist(img.ravel(), 256, [0, 256], color='gray')
plt.axvline(best_t, color='red', linestyle='dashed', label=f'Threshold={best_t}')
plt.title("Histogram & Threshold")
plt.legend()

plt.subplot(1, 3, 3)
plt.imshow(segmented, cmap='gray')
plt.title("Otsu Segmented")
plt.axis("off")

plt.show()
